# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de Google Gemini

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [2]:
from dotenv import load_dotenv
import os
import requests

# Cargar variables de entorno
load_dotenv()
api_key = os.getenv("api_key")

## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

> Se carga el **corpus completo** de 20 News Groups (todas las categorías, subconjunto de entrenamiento) eliminando cabeceras, pies de página y citas.

In [3]:
from sklearn.datasets import fetch_20newsgroups

# Cargar el corpus completo (todas las categorías)
newsgroups_train = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
)

# Construir el corpus omitiendo textos vacíos
corpus = [doc.strip() for doc in newsgroups_train.data if doc.strip()]

print(f"Categorías  : {newsgroups_train.target_names}")
print(f"Documentos  : {len(corpus)}")

Categorías  : ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']
Documentos  : 11014


### 2.2 Transformo a embeddings

> Los embeddings se generan **100% localmente** con `all-MiniLM-L6-v2`.
> No se consume ninguna cuota de API en este paso.

In [4]:
import numpy as np
import math
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

MODELO_LOCAL = "all-MiniLM-L6-v2"
BATCH_SIZE = 64

print(f"Cargando modelo local: {MODELO_LOCAL}")
model = SentenceTransformer(MODELO_LOCAL)
print(f"Modelo cargado.\n")

# Dividir el corpus en lotes
n_docs = len(corpus)
n_batches = math.ceil(n_docs / BATCH_SIZE)
batches = [corpus[i * BATCH_SIZE:(i + 1) * BATCH_SIZE] for i in range(n_batches)]

embeddings_list = []

# Barra de progreso por lotes con tqdm
with tqdm(
    total=n_docs,
    desc="Generando embeddings",
    unit="doc",
    bar_format="{l_bar}{bar:40}{r_bar}"
) as pbar:
    for batch in batches:
        batch_embs = model.encode(batch, convert_to_numpy=True)
        embeddings_list.append(batch_embs)
        pbar.update(len(batch))

embeddings = np.vstack(embeddings_list)

print(f"\nEmbeddings generados correctamente")
print(f"Forma: {embeddings.shape}  →  ({n_docs} docs × {embeddings.shape[1]} dims)")

C:\Users\wwwed\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando modelo local: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12208.94it/s]


Modelo cargado.



Generando embeddings: 100%|████████████████████████████████████████| 11014/11014 [03:12<00:00, 57.15doc/s]


Embeddings generados correctamente
Forma: (11014, 384)  →  (11014 docs × 384 dims)


### 2.3 Creo una query y hago la búsqueda

In [10]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

query = "IDE vs SCSI hard drive speed and controller configuration"
TOP_K = 5

# ── Paso 1: Embedding de la query 
query_embedding = model.encode([query], convert_to_numpy=True)  

# ── Paso 2: Búsqueda por similitud de coseno
similarities = cosine_similarity(query_embedding, embeddings)[0]
top_k_indices = np.argsort(similarities)[::-1][:TOP_K]

print(f"Consulta: '{query}'")
print(f"Dimension del vector de query: {query_embedding.shape}")
print(f"\nTop-{TOP_K} documentos recuperados:")
print(f"{'Rank':<6} | {'ID Doc':<8} | {'Similitud':<10} | {'Fragmento':<50}")
print("-" * 82)

# Lista para armar las filas de tu DataFrame estructurado
data_resultados = []

for rank, idx in enumerate(top_k_indices, 1):
    texto_completo = corpus[idx]
    
    # Guardamos los datos estructurados (recortamos a 1000-1500 chars si los posts son masivos)
    data_resultados.append({
        "Rank": rank,
        "ID_Doc": idx,
        "Similitud": f"{similarities[idx]:.4f}",
        "Texto_Documento": texto_completo[:1200]  # Suficiente contexto para Gemini
    })
    
    # Tu print de consola se queda igual (con el preview corto de 100 chars)
    preview = texto_completo[:100].replace('\n', ' ') + "..."
    print(f"[{rank}]  | #{idx:<7} | {similarities[idx]:.4f}    | {preview}")

# Creamos el DataFrame que usará el prompt de Gemini
results_df = pd.DataFrame(data_resultados)

Consulta: 'IDE vs SCSI hard drive speed and controller configuration'
Dimension del vector de query: (1, 384)

Top-5 documentos recuperados:
Rank   | ID Doc   | Similitud  | Fragmento                                         
----------------------------------------------------------------------------------
[1]  | #1858    | 0.7809    | From article <1qq7i1INNdqc@dns1.NMSU.Edu>, by bgrubb@dante.nmsu.edu (GRUBB):  [Tons of stuff deleted...
[2]  | #4786    | 0.7585    | PLEASE: response directly to me (luoma@binah.cc.brandeis.edu)         by email.  IF there are a suff...
[3]  | #4742    | 0.7564    | Wow, you guys are really going wild on this IDE vs. SCSI thing, and I think it's great!  However, I ...
[4]  | #3277    | 0.7294    | Then don't complain (maybe it wasn't you) that SCSI was so expensive on PC's because all we've had u...
[5]  | #2799    | 0.7175    | {>  {> SCSI-1 {SCSI-2 controller chip; also called SCSI-2 (8-bit)}: 4-6MB/s with  {> 10MB/s burst.  ...


Obtengo los 5 documentos más similares a mi query

In [11]:
from google import genai
import pandas as pd

# ── 1. Initialize the official client ───────────────────────────────────────
client = genai.Client(api_key=api_key)

# ── 2. Build the formatted message (Prompt completely in English) ───────────
# ── 2. Construcción del mensaje formateado (Instrucciones en Español → Respuesta en Inglés) ──
mensaje = f"""
Eres un asistente experto analizando publicaciones históricas del dataset 20 Newsgroups.
A continuación tienes una tabla con los 5 fragmentos de documentos más similares recuperados localmente.

=== CONTEXTO (Resultados de la búsqueda local) ===
{results_df.to_string(index=False)}

=== PREGUNTA DEL USUARIO ===
{query}

Instrucciones críticas para la respuesta:
1. Responde a la pregunta del usuario basándote ESTRICTAMENTE en el contexto provisto.
2. Escribe la respuesta COMPLETAMENTE EN INGLÉS para mantener la concordancia con los documentos.
3. Estructura tu respuesta como una lista numerada que corresponda directamente a los documentos utilizados (ej. [1], [2], [3]...).
4. Para cada documento, detalla de forma clara y concisa qué aporta o qué información provee respecto a la pregunta.
"""

# ── 3. Simple API Call ──────────────────────────────────────────────────────
try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=mensaje,
    )
    
    print(f"\n=== Gemini Response (gemini-2.5-flash) ===")
    print(response.text)

except Exception as e:
    print(f"An error occurred while querying the API: {e}")


=== Gemini Response (gemini-2.5-flash) ===
Here's an analysis of the provided documents concerning IDE vs. SCSI hard drive speed and controller configuration:

1.  **[1858]** This document discusses a user's current setup with a 15ms 210Mb IDE drive (Seagate 1239A) and a standard IDE controller on an ISA 486-50, achieving about 890Kb/sec transfer. The user is considering adding a 300Mb-500Mb SCSI drive for future benefit and asks how the transfer rate would compare with a state-of-the-art SCSI card and hard drive for an ISA PC. It also inquires about the transfer rate achievable with current IDE HDs if a top-of-the-line IDE controller were purchased.

2.  **[4786]** This document addresses the configuration of having both SCSI and IDE hard drives on the same system. The author specifically asks if anyone has successfully achieved this, particularly with SCSI as the boot drive. Information is requested regarding the types of drives used, the SCSI controller, motherboard, BIOS, and any 